In [104]:
import openpyxl
import pandas as pd
import matplotlib.pyplot as plt
from tabulate import tabulate
import numpy as np

### DF_INSIGHTS FUNCTION

This function takes a dataframe as parameter to return valuable insights on the dataframe in a table format using `tabulate` 



In [105]:
def df_insights(df):
    insights = [
        ['Count of Columns', df.shape[1]],
        ['Count of Rows', df.shape[0]],
        ['Missing Values', df.isna().sum().sum()]
        ]

    print(tabulate(insights, headers=["Parameter", "Value"], tablefmt="fancy_grid"))

Importing Datasets

In [106]:
# UV EXPOSURE DATASET

uv_exposure = pd.read_excel('../datasets/uv-county-exposure.xlsx', sheet_name='UV_County_2000-2024', dtype={'COUNTY_FIPS': str})
uv_exposure.rename({'COUNTY NAME': 'COUNTY_NAME', 'STATENAME': 'STATE_NAME'}, axis=1, inplace=True)
uv_exposure['COUNTY_FIPS'] = uv_exposure['COUNTY_FIPS'].str.zfill(5)
uv_exposure = uv_exposure.sort_values(['STATE_NAME', 'COUNTY_NAME'])
uv_exposure




,STATE_NAME,COUNTY_NAME,COUNTY_FIPS,UV_ Wh/m² (2000-2004),UV_ Wh/m² (2005-2009),UV_ Wh/m² (2010-2014),UV_ Wh/m² (2015-2019),UV_ Wh/m² (2020_2024)
2734,Alabama,Autauga,01001,4781.877818,4774.090182,4843.185939,4701.004606,4785.906061
2159,Alabama,Baldwin,01003,4916.409224,4904.892424,4934.852894,4764.348800,4814.548819
565,Alabama,Barbour,01005,4875.885667,4862.169667,4908.160333,4786.408667,4833.696667
974,Alabama,Bibb,01007,4727.518560,4706.299680,4785.613440,4625.073600,4726.646400
2232,Alabama,Blount,01009,4643.034462,4606.273385,4687.476000,4574.497846,4665.236538
...,...,...,...,...,...,...,...,...
1527,Wyoming,Sweetwater,56037,4780.547753,4819.313629,4731.694674,4753.577473,4910.167898
1517,Wyoming,Teton,56039,4294.326031,4282.941613,4162.277026,4303.009508,4529.784895
2234,Wyoming,Uinta,56041,4730.583808,4797.226752,4687.229184,4736.682624,4936.341280
1990,Wyoming,Washakie,56043,4558.186982,4545.855054,4427.304308,4473.671815,4565.537275


In [107]:
df_insights(uv_exposure)

╒══════════════════╤═════════╕
│ Parameter        │   Value │
╞══════════════════╪═════════╡
│ Count of Columns │       8 │
├──────────────────┼─────────┤
│ Count of Rows    │    3107 │
├──────────────────┼─────────┤
│ Missing Values   │       0 │
╘══════════════════╧═════════╛


In [108]:
uv_exposure.isna().sum() * 100/len(uv_exposure)

STATE_NAME               0.0
COUNTY_NAME              0.0
COUNTY_FIPS              0.0
UV_ Wh/m² (2000-2004)    0.0
UV_ Wh/m² (2005-2009)    0.0
UV_ Wh/m² (2010-2014)    0.0
UV_ Wh/m² (2015-2019)    0.0
UV_ Wh/m² (2020_2024)    0.0
dtype: float64

In [109]:
uv_exposure.dtypes

STATE_NAME                   str
COUNTY_NAME                  str
COUNTY_FIPS                  str
UV_ Wh/m² (2000-2004)    float64
UV_ Wh/m² (2005-2009)    float64
UV_ Wh/m² (2010-2014)    float64
UV_ Wh/m² (2015-2019)    float64
UV_ Wh/m² (2020_2024)    float64
dtype: object

In [110]:
# MELANOMA INCIDENCE WITH COUNTY INFO

melanoma_incidence = pd.read_csv('../datasets/melanoma-county-incidence.csv', dtype={'FIPS': str}, skiprows=8, skipfooter=35, engine='python')
melanoma_incidence.drop(columns=['Lower 95% Confidence Interval', 
                                 'Upper 95% Confidence Interval', 
                                 'CI*Rank([rank note])', 
                                 'Lower CI (CI*Rank)', 
                                 'Upper CI (CI*Rank)', 
                                 'Recent 5-Year Trend ([trend note]) in Incidence Rates', 
                                 'Lower 95% Confidence Interval.1', 
                                 'Upper 95% Confidence Interval.1',
                                 'Recent Trend'], inplace=True)
melanoma_incidence.rename({'FIPS': 'COUNTY_FIPS',
                           '2023 Rural-Urban Continuum Codes([rural urban note])': 'RUCC_DESCRIPTION',
                           'Age-Adjusted Incidence Rate([rate note]) - cases per 100,000': 'AGE_RATE_PER100K', 
                           'Average Annual Count': 'AVG_ANNUAL_COUNT'}, axis=1, inplace=True)
melanoma_incidence = melanoma_incidence[melanoma_incidence['COUNTY_FIPS'] != '00000'] # drops the annual agreggate row that has COUNTY_FIPS = '00000'

# SPLIT COUNTY COLUMN INTO COUNTY AND STATE

melanoma_incidence['COUNTY'] = melanoma_incidence['County'].str.split(', ', expand=True)[0]
melanoma_incidence['STATE'] = melanoma_incidence['County'].str.split(', ', expand=True)[1]
melanoma_incidence.insert(0, 'COUNTY', melanoma_incidence.pop('COUNTY'))
melanoma_incidence.insert(1, 'STATE', melanoma_incidence.pop('STATE'))
melanoma_incidence.drop(columns=['County'], inplace=True)

# CLEANING UP COUNTY COLUMN

melanoma_incidence['COUNTY'] = melanoma_incidence['COUNTY'].replace(' County', '', regex=True)

melanoma_incidence = melanoma_incidence.sort_values(['COUNTY_FIPS'])
melanoma_incidence


,COUNTY,STATE,COUNTY_FIPS,RUCC_DESCRIPTION,AGE_RATE_PER100K,AVG_ANNUAL_COUNT
1096,Autauga,Alabama(6),01001,Urban,24,16
217,Baldwin,Alabama(6),01003,Urban,37.2,115
1209,Barbour,Alabama(6),01005,Rural,22.8,8
1909,Bibb,Alabama(6),01007,Urban,15.5,4
1569,Blount,Alabama(6),01009,Urban,19.4,14
...,...,...,...,...,...,...
292,Teton,Wyoming(6),56039,Rural,35.2,10
1836,Uinta,Wyoming(6),56041,Rural,16.4,4
3090,Washakie,Wyoming(6),56043,Rural,*,3 or fewer
3108,Weston,Wyoming(6),56045,Rural,*,3 or fewer


In [111]:
melanoma_incidence.isna().sum() * 100/len(melanoma_incidence)

COUNTY              0.000000
STATE               0.063633
COUNTY_FIPS         0.000000
RUCC_DESCRIPTION    0.000000
AGE_RATE_PER100K    0.000000
AVG_ANNUAL_COUNT    0.000000
dtype: float64

In [112]:
melanoma_incidence[melanoma_incidence['STATE'].isna()]

,COUNTY,STATE,COUNTY_FIPS,RUCC_DESCRIPTION,AGE_RATE_PER100K,AVG_ANNUAL_COUNT
2146,District of Columbia(6),NaN,11001,Urban,10,68
2184,Puerto Rico(6),NaN,72001,Urban,3.3,132


These are the only two rows that have null values in this dataset. <br>
The `COUNTY_FIPS` code for this row is `72001`.<br>
Let's evaluate if there is a UV exposure measurement for this county in the `uv_exposure` df:<br>


In [113]:
if uv_exposure[uv_exposure['COUNTY_FIPS'] == '11001'].empty:
    print('FIPS CODE NOT FOUND')
else:
    print('UV INDEX MEASUREMENT FOUND FOR THIS FIPS CODE')

uv_exposure[uv_exposure['COUNTY_FIPS'] == '11001']

UV INDEX MEASUREMENT FOUND FOR THIS FIPS CODE


,STATE_NAME,COUNTY_NAME,COUNTY_FIPS,UV_ Wh/m² (2000-2004),UV_ Wh/m² (2005-2009),UV_ Wh/m² (2010-2014),UV_ Wh/m² (2015-2019),UV_ Wh/m² (2020_2024)
390,District of Columbia,District of Columbia,11001,4286.810182,4233.748364,4291.252364,4154.570182,4235.43


For this row, we can use `.loc` to use the `COUNTY_FIPS` value to target the row value in the `STATE` column that should be filled in with a valid name:

In [114]:
melanoma_incidence.loc[melanoma_incidence['COUNTY_FIPS'] == '11001', 'STATE'] =  'District of Columbia'

In [115]:
melanoma_incidence.loc[2146]

COUNTY              District of Columbia(6)
STATE                  District of Columbia
COUNTY_FIPS                           11001
RUCC_DESCRIPTION                      Urban
AGE_RATE_PER100K                        10 
AVG_ANNUAL_COUNT                         68
Name: 2146, dtype: str

For the second row with a NaN value, we will follow the same approach:

In [116]:
if uv_exposure[uv_exposure['COUNTY_FIPS'] == '72001' ].empty:
    print('FIPS CODE NOT FOUND')
else:
    print('UV INDEX MEASUREMENT FOUND FOR THIS FIPS CODE')


FIPS CODE NOT FOUND


Since there is no measurement for this county, we can drop this row from the `melanoma_incidence` dataset.

In [117]:
melanoma_incidence.dropna(subset=['STATE'], inplace=True)
melanoma_incidence.isna().sum()

COUNTY              0
STATE               0
COUNTY_FIPS         0
RUCC_DESCRIPTION    0
AGE_RATE_PER100K    0
AVG_ANNUAL_COUNT    0
dtype: int64

In [118]:
melanoma_incidence.dtypes

COUNTY              str
STATE               str
COUNTY_FIPS         str
RUCC_DESCRIPTION    str
AGE_RATE_PER100K    str
AVG_ANNUAL_COUNT    str
dtype: object

In [119]:
# RURAL URBAN CONTINUUM CODES

continuum_codes = pd.read_excel('../datasets/rural-urban-continuum-codes-2023.xlsx')
continuum_codes

,FIPS,State,County_Name,Population_2020,RUCC_2023,Description
0,1001,AL,Autauga County,58805,2.0,"Metro - Counties in metro areas of 250,000 to ..."
1,1003,AL,Baldwin County,231767,3.0,Metro - Counties in metro areas of fewer than ...
2,1005,AL,Barbour County,25223,6.0,"Nonmetro - Urban population of 5,000 to 20,000..."
3,1007,AL,Bibb County,22293,1.0,Metro - Counties in metro areas of 1 million p...
4,1009,AL,Blount County,59134,1.0,Metro - Counties in metro areas of 1 million p...
...,...,...,...,...,...,...
3230,72151,PR,Yabucoa Municipio,30426,1.0,Metro - Counties in metro areas of 1 million p...
3231,72153,PR,Yauco Municipio,34172,2.0,"Metro - Counties in metro areas of 250,000 to ..."
3232,78010,VI,St. Croix Island,41004,5.0,"Nonmetro - Urban population of 20,000 or more,..."
3233,78020,VI,St. John Island,3881,9.0,"Nonmetro - Urban population of fewer than 5,00..."


In [120]:
continuum_codes.isna().sum()

FIPS               0
State              0
County_Name        0
Population_2020    0
RUCC_2023          2
Description        0
dtype: int64

In [121]:
continuum_codes[continuum_codes['RUCC_2023'].isna()]

,FIPS,State,County_Name,Population_2020,RUCC_2023,Description
3146,60030,AS,Rose Island,0,NaN,Not Applicable
3147,60040,AS,Swains Island,0,NaN,Not Applicable


These two rows have null values for Rural Urban Continuum Codes. <br>
Here, I will drop these two rows that are ireelevant for our analysis, since the population in these counties located in the American Samoa is zero.

In [122]:
continuum_codes.dropna(subset=['RUCC_2023'], inplace=True)
continuum_codes.isna().sum()

FIPS               0
State              0
County_Name        0
Population_2020    0
RUCC_2023          0
Description        0
dtype: int64

In [123]:
# SEER*STATS

seer_db = pd.read_csv('../datasets/seer-stats.csv')
seer_db

,Rural-Urban Continuum Code,Age recode with single ages and 90+,Sex,Year of diagnosis,"Race and origin recode (NHW, NHB, NHAIAN, NHAPI, Hispanic)",Site recode ICD-O-3 2023 Revision Expanded,"ICD-O-3 Hist/behav, malignant",Behavior code ICD-O-3,Combined Summary Stage with Expanded Regional Codes (2004+)
0,Counties in metropolitan areas ge 1 million pop,70 years,Male,2021,Non-Hispanic White,Melanoma Of The Skin,"8720/3: Malignant melanoma, NOS",Malignant,Localized only
1,Counties in metropolitan areas ge 1 million pop,61 years,Female,2017,Non-Hispanic White,Melanoma Of The Skin,"8720/3: Malignant melanoma, NOS",Malignant,Localized only
2,Counties in metropolitan areas ge 1 million pop,63 years,Female,2019,Non-Hispanic White,Melanoma Of The Skin,8743/3: Superficial spreading melanoma,Malignant,Localized only
3,Counties in metropolitan areas ge 1 million pop,58 years,Male,2019,Non-Hispanic White,Melanoma Of The Skin,8743/3: Superficial spreading melanoma,Malignant,Localized only
4,Counties in metropolitan areas ge 1 million pop,72 years,Male,2018,Non-Hispanic White,Melanoma Of The Skin,8743/3: Superficial spreading melanoma,Malignant,Localized only
...,...,...,...,...,...,...,...,...,...
156852,Counties in metropolitan areas of lt 250 thous...,73 years,Male,2022,Non-Hispanic White,Melanoma Of The Skin,"8720/3: Malignant melanoma, NOS",Malignant,Unknown/unstaged/unspecified/DCO
156853,Counties in metropolitan areas of lt 250 thous...,83 years,Male,2022,Non-Hispanic White,Melanoma Of The Skin,"8720/3: Malignant melanoma, NOS",Malignant,Unknown/unstaged/unspecified/DCO
156854,Counties in metropolitan areas ge 1 million pop,81 years,Female,2022,Non-Hispanic White,Melanoma Of The Skin,"8720/3: Malignant melanoma, NOS",Malignant,Unknown/unstaged/unspecified/DCO
156855,Nonmetropolitan counties adjacent to a metropo...,69 years,Male,2022,Non-Hispanic White,Melanoma Of The Skin,8721/3: Nodular melanoma,Malignant,Regional by direct extension only


In [124]:
seer_db.isna().sum()

Rural-Urban Continuum Code                                     0
Age recode with single ages and 90+                            0
Sex                                                            0
Year of diagnosis                                              0
Race and origin recode (NHW, NHB, NHAIAN, NHAPI, Hispanic)     0
Site recode ICD-O-3 2023 Revision Expanded                     0
ICD-O-3 Hist/behav, malignant                                  0
Behavior code ICD-O-3                                          0
Combined Summary Stage with Expanded Regional Codes (2004+)    0
dtype: int64